# **Router Pattern in LangGraph**

A **router** uses structured LLM output to classify the user's intent, then a conditional edge sends execution to the matching node.

Flow: `START → decider_node → (condition fn) → create_insta | create_twitter | create_linkedin → END`

In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

if os.environ.get('GROQ_API_KEY'):
    print("Groq API Key is set.")
else:
    raise ValueError("GROQ_API_KEY not found.")

In [ ]:
llm = ChatGroq(model="llama3-8b-8192", temperature=0)

### **LLM Pydantic Schema**
Forcing the LLM to output a structured object means we can reliably extract `category` and `topic` without parsing free text.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class llm_schema(BaseModel):
    # Literal constrains the value to exactly one of these strings — perfect for routing
    category: Literal['insta', 'twitter', 'linkedin'] = Field(..., description="Category of the post to generate")
    topic:    str                                      = Field(..., description="Topic of the post to generate")

# with_structured_output wraps the LLM so it always returns an llm_schema object
llm_with_schema = llm.with_structured_output(llm_schema)

In [ ]:
# Quick test: does the LLM correctly classify this request?
llm_with_schema.invoke("I want to generate a post for twitter about AI").category

### **State Schema**

In [ ]:
from typing import TypedDict, List

class graph_schema(TypedDict):
    input:    str   # raw user request, e.g. "I want a LinkedIn post about AI"
    topic:    str   # extracted by decider_node
    post:     str   # filled by the chosen creator node
    category: str   # 'insta' | 'twitter' | 'linkedin' — extracted by decider_node

### **Node Functions**

In [ ]:
def decider_node(state: graph_schema) -> graph_schema:
    """Classifies the user's request into a category and extracts the topic."""

    user_input = state['input']

    # llm_with_schema returns an llm_schema Pydantic object — never raw text
    response = llm_with_schema.invoke(user_input)

    state['category'] = response.category   # 'insta', 'twitter', or 'linkedin'
    state['topic']    = response.topic      # the extracted subject matter

    return state


def create_post_insta(state: graph_schema) -> graph_schema:
    topic = state['topic']
    post  = llm.invoke(f"Write an Instagram post about {topic}. Keep the tone casual and engaging.").content
    return {'post': post}


def create_post_twitter(state: graph_schema) -> graph_schema:
    topic = state['topic']
    post  = llm.invoke(f"Write a Twitter post about {topic}. Keep the tone quick.").content
    return {'post': post}


def create_post_linkedin(state: graph_schema) -> graph_schema:
    topic = state['topic']
    post  = llm.invoke(f"Write a LinkedIn post about {topic}. Keep the tone professional and informative.").content
    return {'post': post}

### **Conditional Edge Function**
This function reads the category set by `decider_node` and returns a string that LangGraph maps to the next node.

In [ ]:
def condition(state: graph_schema) -> str:
    """Returns the key that LangGraph uses to pick the next node."""
    category = state['category']

    if   category == 'insta':    return 'create_insta'
    elif category == 'twitter':  return 'create_twitter'
    elif category == 'linkedin': return 'create_linkedin'
    else:
        raise ValueError(f"Unknown category: {category}")

### **Build the Graph**

In [ ]:
from langgraph.graph import StateGraph, START, END
from IPython.display import Image

graph = StateGraph(graph_schema)

graph.add_node("decider",        decider_node)
graph.add_node("create_insta",   create_post_insta)
graph.add_node("create_twitter", create_post_twitter)
graph.add_node("create_linkedin",create_post_linkedin)

graph.add_edge(START, "decider")

# add_conditional_edges wires the condition function + a mapping of return values → nodes
graph.add_conditional_edges("decider", condition, {
    'create_insta':    "create_insta",
    'create_twitter':  "create_twitter",
    'create_linkedin': "create_linkedin",
})

graph.add_edge("create_insta",    END)
graph.add_edge("create_twitter",  END)
graph.add_edge("create_linkedin", END)

route_graph = graph.compile()
Image(route_graph.get_graph().draw_mermaid_png())

In [ ]:
route_graph.invoke({
    "input":    "I want to generate a post for twitter about AI",
    "topic":    "",
    "category": "",
    "post":     ""
})